# Procedure

Following this file, one can recreate our results:
- if one wants to recreate it from scratch, the 1830s section from the COHA is necessary. It must be structured identical to the Example Corpus, the folder called 'COHA'. Delete 'preprocessed_corpus.json', 'preprocessed_filtered_corpus.json' and all non-python files from the 'Embeddings' folder - otherwise the preexisting files will be used.
- if one wants to recreate the computing of the embeddings and anything after, no additional data is needed. Make sure 'preprocessed_corpus.json' and 'preprocessed_filtered_corpus.json' are in place. Delete any non-python files from the 'Embeddings' folder.
- if one only wants to recreate the computing of the historical VAD lexica, no additional data is needed and no original files should be deleted.

Note: if you decide to retrain the embeddings, results for SGNS, CBOW, and FastText will slightly differ due to the non-deterministic nature of these algorithms.

In [2]:
from model_definition import Model
from pathlib import Path
from correlation import calculate_correlation, compare_lexica
import pandas as pd
import random

# Global Variables

In [3]:
list_of_embeddings = ['SGNS', 'CBOW', 'FastText', 'PPMI', 'SVD_PPMI']
list_of_induction_algorithms = ['kNN', 'PaRaSim', 'RandomWalk', 'LinearRegression']

# Section 4.1

# The VAD lexica

For our experiments we need to limit the VAD lexica to the vocab of the COHA and take out the GOLD standard words. In addition, to follow Hellrich et al., we have to get the words from ANEW but the values from Warriner. And as a comparison, we need to get the words from ANEW and the values from NRC VAD.

# Filtering out Gold Standard Words and Words not in the COHA

In [7]:
gold_standard_path = Path('VADLexica/goldEN.vad')
lemmata_in_COHA_path = Path('Embeddings/AAA_word_list.txt')

anew_path = Path('VADLexica/ANEW')
anew_filtered_path = Path('VADLexica/ANEW_refined_withoutGOLD_limited_to_COHA')

warriner_path = Path('VADLexica/Warriner_et_alemotratings.csv')
warriner_filtered_path = Path('VADLexica/Warriner_refined_withoutGOLD_limited_to_COHA')

nrc_vad_path = Path('VADLexica/NRC-VAD-Lexicon.txt')
nrc_vad_filtered_path = Path('VADLexica/NRC_VAD_refined_withoutGOLD_limited_to_COHA')

warriner_path_final_version = Path('VADLexica/Warriner_ANEW_refined_withoutGOLD_limited_to_COHA')
nrc_vad_path_final_version = Path('VADLexica/NRC_VAD_ANEW_refined_withoutGOLD_limited_to_COHA')

In [3]:
GOLDSTANDARD = []
with open(gold_standard_path) as f:
    for line in f:
        GOLDSTANDARD.append(line.split()[0])
Lemmata_in_COHA = []
with open(lemmata_in_COHA_path) as f:
    for line in f:
        Lemmata_in_COHA.append(line.strip())

In [4]:
GOLDSTANDARD = []
with open(gold_standard_path) as f:
    for line in f:
        GOLDSTANDARD.append(line.split()[0])
Lemmata_in_COHA = []
with open(lemmata_in_COHA_path) as f:
    for line in f:
        Lemmata_in_COHA.append(line.strip())

Anew = []
with open(anew_path, mode = 'r') as f:
    for line in f:
        words = line.split()
        Anew.append([words[0], words[2], words[4], words[6]])
with open(anew_filtered_path, mode = 'w') as f:
    for row in Anew:
        if row[0] in Lemmata_in_COHA and row[0] not in GOLDSTANDARD:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

Warriner = []
with open(warriner_path, mode = 'r') as f:
    for line in f:
        words = line.split(',')
        Warriner.append([words[1], words[2], words[5], words[8]])
with open(warriner_filtered_path, mode = 'w') as f:
    for row in Warriner:
        if row[0] in Lemmata_in_COHA and row[0] not in GOLDSTANDARD:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

NRC_VAD = []
with open(nrc_vad_path, mode = 'r') as f:
    for line in f:
        words = line.split('\t')
        NRC_VAD.append([words[0], round(float(words[1])*8+1, 3), round(float(words[2])*8+1,3), round(float(words[3])*8+1, 3)])
with open(nrc_vad_filtered_path, mode = 'w') as f:
    for row in NRC_VAD:
        if row[0] in Lemmata_in_COHA and row[0] not in GOLDSTANDARD:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')


# Set to ANEW

In [6]:
Anew = []
Warriner = []
NRC_VAD = []

with open(anew_filtered_path) as f:
    for line in f:
        Anew.append(line.split('\t')[0])
with open(warriner_filtered_path) as f:
    for line in f:
        Warriner.append(line.split('\t'))
with open(nrc_vad_filtered_path) as f:
    for line in f:
        NRC_VAD.append(line.split('\t'))

with open(warriner_path_final_version, mode = 'w') as f:
    for row in Warriner:
        if row[0] in Anew:
            line = '\t'.join(map(str, row))
            f.write(line)
with open(nrc_vad_path_final_version, mode = 'w') as f:
    for row in NRC_VAD:
        if row[0] in Anew:
            line = '\t'.join(map(str, row))
            f.write(line)

# The Preprocessing

As the preprocessed corpus is saved, it is sufficient to call the preprocessing method once (best with 'PPMI' or 'SVD_PPMI' to obtain both the filtered and unfiltered version). After that, every new Model object will load the correct corpus without having to preprocess it again.

In [ ]:
model = Model('SVD_PPMI', 'kNN', warriner_path_final_version, 'Warriner') #kNN and Warriner were chosen arbitrarily 
model.preprocess_corpus()
print(f"Is the corpus preprocessed now? {model.corpus_is_preprocessed}")

# The Training of the Embeddings

Similarly, each embedding has to be trained only once. Afterward, any new model will load the correct embedding.

In [ ]:
for embedding in list_of_embeddings:
    model = Model(embedding, 'kNN', warriner_path_final_version, 'Warriner') #kNN and Warriner were chosen arbitrarily 
    model.train_embedding()
    print(model.embedding_is_computed)

# Sections 4.2 and 4.3

# Computing all the different historical VAD lexica

In [5]:
list_of_lexicon_paths_and_names = [(warriner_path_final_version, 'Warriner' ), (nrc_vad_path_final_version, 'NRC_VAD')]

In [ ]:
for embedding in list_of_embeddings:
    for induction_algorithm in list_of_induction_algorithms:
        for lexicon_path_and_name in list_of_lexicon_paths_and_names:
            print(embedding)
            print(induction_algorithm)
            print(lexicon_path_and_name[1])
            model = Model(embedding, induction_algorithm, *lexicon_path_and_name)
            model.induce_historical_vad_lexicon()

# Computing Correlation for the historical VAD lexica 

In [9]:
column_names = ['r_V', 'p_value_V', 'r_A', 'p_value_A', 'r_D', 'p_value_D', 'r_mean']
index_names = ['kNN CBOW', 'kNN FastText', 'kNN PPMI', 'kNN SGNS', 'kNN SVD_PPMI',
               'PaRaSim CBOW', 'PaRaSim FastText', 'PaRaSim PPMI', 'PaRaSim SGNS', 'PaRaSim SVD_PPMI',
               'RandomWalk CBOW', 'RandomWalk FastText', 'RandomWalk PPMI', 'RandomWalk SGNS', 'RandomWalk SVD_PPMI',
               'LinReg CBOW', 'LinReg FastText', 'LinReg PPMI', 'LinReg SGNS', 'LinReg SVD_PPMI']

In [7]:
warriner_files= []
for induction_algorithm in list_of_induction_algorithms:
    folder = Path('HistoricalVAD/'+ induction_algorithm)
    warriner_files.extend(list(folder.glob('Warriner*')))
warriner_correlation = [calculate_correlation(gold_standard_path, path) for path in warriner_files]
warriner_correlation_df = pd.DataFrame(warriner_correlation, columns=column_names, index=index_names)

In [8]:
nrc_vad_files= []
for induction_algorithm in list_of_induction_algorithms:
    folder = Path('HistoricalVAD/'+ induction_algorithm)
    nrc_vad_files.extend(list(folder.glob('NRC_VAD*')))
nrc_vad_correlation = [calculate_correlation(gold_standard_path, path) for path in nrc_vad_files]
nrc_vad_correlation_df = pd.DataFrame(nrc_vad_correlation, columns=column_names, index=index_names)


# Results for Warriner based lexica (Section 4.2)

In [9]:
warriner_correlation_df

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
kNN CBOW,0.4706,0.0000,0.3479,0.0004,0.1688,0.0933,0.3291
kNN FastText,0.5391,0.0000,0.3730,0.0001,0.2430,0.0149,0.3850
kNN PPMI,0.4543,0.0000,0.3223,0.0011,0.2342,0.0190,0.3370
kNN SGNS,0.4786,0.0000,0.4154,0.0000,0.1991,0.0471,0.3643
kNN SVD_PPMI,0.4227,0.0000,0.3267,0.0009,0.1498,0.1369,0.2997
PaRaSim CBOW,0.2511,0.0117,0.1908,0.0572,0.2017,0.0442,0.2145
PaRaSim FastText,0.4791,0.0000,0.3826,0.0001,0.1739,0.0836,0.3452
PaRaSim PPMI,0.4743,0.0000,0.3840,0.0001,0.1835,0.0676,0.3473
PaRaSim SGNS,0.5223,0.0000,0.4035,0.0000,0.2020,0.0439,0.3759
PaRaSim SVD_PPMI,0.1236,0.2203,-0.0542,0.5921,0.0942,0.3512,0.0545


# Results for NRC-VAD based lexica (Section 4.3)

In [10]:
nrc_vad_correlation_df

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
kNN CBOW,0.4750,0.0000,0.3612,0.0002,0.0741,0.4637,0.3034
kNN FastText,0.4871,0.0000,0.4641,0.0000,0.0332,0.7433,0.3281
kNN PPMI,0.5110,0.0000,0.3832,0.0001,-0.0044,0.9657,0.2966
kNN SGNS,0.5225,0.0000,0.3978,0.0000,0.1122,0.2663,0.3442
kNN SVD_PPMI,0.4625,0.0000,0.4182,0.0000,0.0113,0.9114,0.2973
PaRaSim CBOW,0.2191,0.0285,0.2202,0.0277,0.0381,0.7067,0.1591
PaRaSim FastText,0.4794,0.0000,0.4010,0.0000,0.0608,0.5482,0.3137
PaRaSim PPMI,0.4936,0.0000,0.3831,0.0001,0.0422,0.6770,0.3063
PaRaSim SGNS,0.5397,0.0000,0.4029,0.0000,0.0978,0.3332,0.3468
PaRaSim SVD_PPMI,0.0379,0.7084,-0.0762,0.4512,-0.0087,0.9318,-0.0157


# Section 4.4

Here the seed word lexica of increasing size are generated (with random subsamples)

In [ ]:
def get_sub_lexica(n_of_w, lexicon_path, lexicon_name, save_dir = 'VADLexica/'):
    with open(lexicon_path) as f:
        doc = f.readlines()
        range_int = len(doc)

    base_lexicon = []
    with open(lexicon_path) as f:
        for l in f:
            l = l.split('\t')
            base_lexicon.append([l[0], float(l[1]), float(l[2]), float(l[3])])

    random.seed(42)
    for i in range(50):
        list1 = random.sample(range(0, range_int), k=n_of_w)
        result_lexicon = [base_lexicon[i] for i in list1]
        
        save_dir = Path(save_dir + lexicon_name + str(n_of_w) + '/')
        save_dir.mkdir(parents=True, exist_ok=True)
        
        save_path = save_dir / f'{lexicon_name}_{n_of_w}_{i+1}'
        with open(save_path,mode='w') as f:
            for row in result_lexicon:
                l = '\t'.join(map(str, row))
                f.write(l + '\n')

We need the Warriner/NRC-VAD lexica that only have COHA words, but no gold standard words. We computed them above, see Section 4.1.

In [35]:
# for warriner_filtered_path ('VADLexica/Warriner_refined_withoutGOLD_limited_to_COHA')
for i in [2000, 3000, 4000, 5000, 6000, 7000, 8000]:
    get_sub_lexica(i, warriner_filtered_path, 'Warriner')

In [36]:
# nrc_vad_filtered_path ('VADLexica/NRC_VAD_refined_withoutGOLD_limited_to_COHA')
for i in [2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]:
    get_sub_lexica(i, nrc_vad_filtered_path, 'NRC_VAD')

Now we need to compute all historical VAD lexica. We only do this with the three models that performed best in Sections 4.2 and 4.3.

In [7]:
best_models_warriner = [('FastText', 'kNN'), ('FastText', 'LinearRegression'), ('SGNS', 'LinearRegression')]
best_models_nrc_vad = [('FastText', 'LinearRegression'),('SGNS', 'LinearRegression'), ('SGNS', 'PaRaSim')]

In [ ]:
def compute_all(lexicon_name, best_models, folder = 'VADLexica/'):
    lexicon_folder_level = []
    folder = Path(folder)
    lexicon_folder_level.extend(sorted(list(folder.glob(f'{lexicon_name}[0-9]*'))))
    print(lexicon_folder_level)
    
    for lexicon_folder in lexicon_folder_level:
        print(lexicon_folder)
        lexicon_level = []
        lexicon_level.extend(sorted(list(lexicon_folder.glob(f'{lexicon_name}*'))))
        for path in lexicon_level:
            name = str(path.parts[2])
            for m in best_models:
                model = Model(*m, path, f'{name}')
                save_path = f"{m[0]}_{m[1]}/{path.parts[1]}"
                print(save_path)
                print(name)
                model.induce_historical_vad_lexicon(save_path)

In [ ]:
compute_all('Warriner', best_models_warriner)

In [ ]:
compute_all('NRC_VAD', best_models_nrc_vad)

 Let us compute all Correlations and the averages of the Correlations

In [23]:
def compare_all(lexicon_name, best_models, gold_standard_path= gold_standard_path, save_folder= 'HistoricalVAD'):
    results = []
    for m in best_models:
        lexicon_folder_level = []
        folder = Path(f'{save_folder}/{m[0]}_{m[1]}/')
        lexicon_folder_level.extend(sorted(list(folder.glob(f'{lexicon_name}[0-9]*'))))

        for lexicon_folder in lexicon_folder_level:
            results.append(compare_lexica(lexicon_folder, gold_standard_path))
    return results

In [15]:
results_warriner = compare_all('Warriner', best_models_warriner)

index_names_warriner = [best_models_warriner[0][0]+best_models_warriner[0][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[1][0]+best_models_warriner[1][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[2][0]+best_models_warriner[2][1]+str(i*1000) for i in range(2,9)]

results_warriner_df = pd.DataFrame(results_warriner, columns=column_names, index=index_names_warriner) 
results_warriner_df

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextkNN2000,0.433120,0.000208,0.318086,0.008218,0.168552,0.166494,0.306590
FastTextkNN3000,0.448616,0.000040,0.351348,0.002086,0.191316,0.098672,0.330424
FastTextkNN4000,0.467010,0.000004,0.362154,0.000976,0.208002,0.058976,0.345730
FastTextkNN5000,0.495046,0.000000,0.391458,0.000146,0.238648,0.027396,0.375044
FastTextkNN6000,0.490898,0.000000,0.419850,0.000022,0.260940,0.015552,0.390560
FastTextkNN7000,0.485060,0.000000,0.434950,0.000006,0.265996,0.009784,0.395336
FastTextkNN8000,0.483036,0.000000,0.440278,0.000000,0.251774,0.012520,0.391698
FastTextLinearRegression2000,0.493096,0.000000,0.378768,0.000968,0.229132,0.041308,0.367000
FastTextLinearRegression3000,0.518500,0.000000,0.419766,0.000048,0.241226,0.020514,0.393178
FastTextLinearRegression4000,0.528978,0.000000,0.427164,0.000022,0.239472,0.019338,0.398538


In [16]:
results_warriner_df.iloc[:7].to_csv(Path('HistoricalVAD/Correlations/Summary_FastText_kNN_Warriner.csv'))
results_warriner_df.iloc[7:14].to_csv(Path('HistoricalVAD/Correlations/Summary_FastText_LinReg_Warriner.csv')) 
results_warriner_df.iloc[14:].to_csv(Path('HistoricalVAD/Correlations/Summary_SGNS_LinReg_Warriner.csv')) 

In [23]:
results_nrc_vad = compare_all('NRC_VAD', best_models_nrc_vad)

index_names_nrc_vad_0 = [best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'10000']+[best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'0'+str(i*1000) for i in range(2,10)]
index_names_nrc_vad_1 = [best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'10000']+[best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'0'+str(i*1000) for i in range(2,10)] 
index_names_nrc_vad_2 = [best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'10000']+[best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'0'+str(i*1000) for i in range(2,10)] 

index_names_nrc_vad = index_names_nrc_vad_0 + index_names_nrc_vad_1 + index_names_nrc_vad_2

results_nrc_vad_df = pd.DataFrame(results_nrc_vad, columns=column_names, index=index_names_nrc_vad) 
results_nrc_vad_df 

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextLinearRegression10000,0.579294,0.0,0.452906,0.000000,0.225348,0.024298,0.419180
FastTextLinearRegression02000,0.535510,0.0,0.410920,0.000116,0.208544,0.048144,0.384994
FastTextLinearRegression03000,0.549906,0.0,0.422730,0.000032,0.211336,0.041018,0.394660
FastTextLinearRegression04000,0.562234,0.0,0.432832,0.000006,0.215132,0.036008,0.403402
FastTextLinearRegression05000,0.569170,0.0,0.440802,0.000004,0.221810,0.029446,0.410592
FastTextLinearRegression06000,0.578146,0.0,0.449608,0.000000,0.223628,0.026608,0.417118
FastTextLinearRegression07000,0.574346,0.0,0.446344,0.000000,0.222234,0.027424,0.414302
FastTextLinearRegression08000,0.576388,0.0,0.448926,0.000000,0.222650,0.026720,0.415984
FastTextLinearRegression09000,0.579076,0.0,0.449960,0.000000,0.223534,0.026034,0.417524
SGNSLinearRegression10000,0.574530,0.0,0.494756,0.000000,0.206910,0.039446,0.425400


In [26]:
results_nrc_vad_df.iloc[:9].to_csv(Path('HistoricalVAD/Correlations/Summary_FastText_LinReg_NRC_VAD.csv'))
results_nrc_vad_df.iloc[9:18].to_csv(Path('HistoricalVAD/Correlations/Summary_SGNS_LinReg_NRC_VAD.csv')) 
results_nrc_vad_df.iloc[18:].to_csv(Path('HistoricalVAD/Correlations/Summary_SGNS_PaRaSim_NRC_VAD.csv')) 

# Section 4.5

1) Warriner based lexica evaluated against NRC-VAD valued stable words

In [4]:
NRC_stable_path = Path('stable_words/stable_with_NRC.txt')

In [ ]:
NRC_stable = []
with open(NRC_stable_path) as f:
    for line in f:
        NRC_stable.append(line.split()[0])
Lemmata_in_COHA = []
with open(lemmata_in_COHA_path) as f:
    for line in f:
        Lemmata_in_COHA.append(line.strip())

Anew = []
with open(anew_path, mode = 'r') as f:
    for line in f:
        words = line.split()
        Anew.append([words[0], words[2], words[4], words[6]])
with open(anew_filtered_path, mode = 'w') as f:
    for row in Anew:
        if row[0] in Lemmata_in_COHA and row[0] not in NRC_stable:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

Warriner = []
with open(warriner_path, mode = 'r') as f:
    for line in f:
        words = line.split(',')
        Warriner.append([words[1], words[2], words[5], words[8]])
with open(warriner_filtered_path, mode = 'w') as f:
    for row in Warriner:
        if row[0] in Lemmata_in_COHA and row[0] not in NRC_stable:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

NRC_VAD = []
with open(nrc_vad_path, mode = 'r') as f:
    for line in f:
        words = line.split('\t')
        NRC_VAD.append([words[0], round(float(words[1])*8+1, 3), round(float(words[2])*8+1,3), round(float(words[3])*8+1, 3)])
with open(nrc_vad_filtered_path, mode = 'w') as f:
    for row in NRC_VAD:
        if row[0] in Lemmata_in_COHA and row[0] not in NRC_stable:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

In [ ]:
for i in [2000, 3000, 4000, 5000, 6000, 7000, 8000]:
    get_sub_lexica(i, warriner_filtered_path, 'Warriner', save_dir = 'stable_words/')

In [ ]:
compute_all('Warriner', best_models_warriner, folder = 'stable_words/')

In [30]:
results_warriner = compare_all('Warriner', best_models_warriner, gold_standard_path=NRC_stable_path, save_folder = 'stable_words')

index_names_warriner = [best_models_warriner[0][0]+best_models_warriner[0][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[1][0]+best_models_warriner[1][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[2][0]+best_models_warriner[2][1]+str(i*1000) for i in range(2,9)]

results_warriner_df = pd.DataFrame(results_warriner, columns=column_names, index=index_names_warriner) 
results_warriner_df

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextkNN2000,0.723412,0.000002,0.636814,0.000210,0.389534,0.027054,0.583250
FastTextkNN3000,0.733874,0.000000,0.668046,0.000020,0.373644,0.032056,0.591860
FastTextkNN4000,0.745354,0.000000,0.681520,0.000002,0.364092,0.032416,0.596992
FastTextkNN5000,0.758196,0.000000,0.708322,0.000000,0.360464,0.033162,0.608996
FastTextkNN6000,0.766264,0.000000,0.726158,0.000000,0.372080,0.024116,0.621498
FastTextkNN7000,0.765610,0.000000,0.736132,0.000000,0.375152,0.021206,0.625628
FastTextkNN8000,0.754642,0.000000,0.734976,0.000000,0.364324,0.023608,0.617972
FastTextLinearRegression2000,0.824562,0.000000,0.561570,0.000702,0.506514,0.001856,0.630870
FastTextLinearRegression3000,0.844176,0.000000,0.592270,0.000128,0.515680,0.001146,0.650714
FastTextLinearRegression4000,0.848138,0.000000,0.604550,0.000074,0.525462,0.000808,0.659380


In [25]:
results_warriner_df.iloc[:7].to_csv(Path('stable_words/Correlations/Summary_FastText_kNN_Warriner.csv'))
results_warriner_df.iloc[7:14].to_csv(Path('stable_words/Correlations/Summary_FastText_LinReg_Warriner.csv')) 
results_warriner_df.iloc[14:].to_csv(Path('stable_words/Correlations/Summary_SGNS_LinReg_Warriner.csv')) 

2) NRC-VAD based lexica evaluated against Warriner valued stable words

In [26]:
Warriner_stable_path = Path('stable_words/stable_with_Warriner.txt')

In [ ]:
Warriner_stable = []
with open(Warriner_stable_path) as f:
    for line in f:
        Warriner_stable.append(line.split()[0])
Lemmata_in_COHA = []
with open(lemmata_in_COHA_path) as f:
    for line in f:
        Lemmata_in_COHA.append(line.strip())

Anew = []
with open(anew_path, mode = 'r') as f:
    for line in f:
        words = line.split()
        Anew.append([words[0], words[2], words[4], words[6]])
with open(anew_filtered_path, mode = 'w') as f:
    for row in Anew:
        if row[0] in Lemmata_in_COHA and row[0] not in Warriner_stable:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

Warriner = []
with open(warriner_path, mode = 'r') as f:
    for line in f:
        words = line.split(',')
        Warriner.append([words[1], words[2], words[5], words[8]])
with open(warriner_filtered_path, mode = 'w') as f:
    for row in Warriner:
        if row[0] in Lemmata_in_COHA and row[0] not in Warriner_stable:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

NRC_VAD = []
with open(nrc_vad_path, mode = 'r') as f:
    for line in f:
        words = line.split('\t')
        NRC_VAD.append([words[0], round(float(words[1])*8+1, 3), round(float(words[2])*8+1,3), round(float(words[3])*8+1, 3)])
with open(nrc_vad_filtered_path, mode = 'w') as f:
    for row in NRC_VAD:
        if row[0] in Lemmata_in_COHA and row[0] not in Warriner_stable:
            line = '\t'.join(map(str, row))
            f.write(line + '\n')

In [ ]:
for i in [2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]:
    get_sub_lexica(i, nrc_vad_filtered_path, 'NRC_VAD', save_dir = 'stable_words/')

In [ ]:
compute_all('NRC_VAD', best_models_nrc_vad, folder = 'stable_words/')

In [28]:
results_nrc_vad = compare_all('NRC_VAD', best_models_nrc_vad, Warriner_stable_path, save_folder = 'stable_words')

index_names_nrc_vad_0 = [best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'10000']+[best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'0'+str(i*1000) for i in range(2,10)]
index_names_nrc_vad_1 = [best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'10000']+[best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'0'+str(i*1000) for i in range(2,10)] 
index_names_nrc_vad_2 = [best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'10000']+[best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'0'+str(i*1000) for i in range(2,10)] 

index_names_nrc_vad = index_names_nrc_vad_0 + index_names_nrc_vad_1 + index_names_nrc_vad_2

results_nrc_vad_df = pd.DataFrame(results_nrc_vad, columns=column_names, index=index_names_nrc_vad) 
results_nrc_vad_df 

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextLinearRegression10000,0.873566,0.0,0.473910,0.002710,0.416260,0.009440,0.587906
FastTextLinearRegression02000,0.838592,0.0,0.428460,0.010864,0.392670,0.024830,0.553236
FastTextLinearRegression03000,0.849386,0.0,0.451178,0.006414,0.397140,0.017642,0.565906
FastTextLinearRegression04000,0.860386,0.0,0.460876,0.004548,0.400606,0.016508,0.573960
FastTextLinearRegression05000,0.863016,0.0,0.463790,0.003814,0.407794,0.012658,0.578208
FastTextLinearRegression06000,0.865712,0.0,0.469512,0.003440,0.411198,0.011576,0.582142
FastTextLinearRegression07000,0.868390,0.0,0.475102,0.002772,0.411646,0.011018,0.585046
FastTextLinearRegression08000,0.871832,0.0,0.472944,0.002906,0.416184,0.010020,0.586978
FastTextLinearRegression09000,0.871766,0.0,0.472816,0.002850,0.413150,0.010244,0.585908
SGNSLinearRegression10000,0.888360,0.0,0.467968,0.003082,0.542148,0.000452,0.632816


In [29]:
results_nrc_vad_df.iloc[:9].to_csv(Path('stable_words/Correlations/Summary_FastText_LinReg_NRC_VAD.csv'))
results_nrc_vad_df.iloc[9:18].to_csv(Path('stable_words/Correlations/Summary_SGNS_LinReg_NRC_VAD.csv')) 
results_nrc_vad_df.iloc[18:].to_csv(Path('stable_words/Correlations/Summary_SGNS_PaRaSim_NRC_VAD.csv')) 

# Appendix

Careful, this will overwrite the HistoricalVAD/Correlations!

In [4]:
gold_standard_path_with_warriner_scores = Path('pseudo-gold/goldEN_with_Warriner.vad')
gold_standard_path_with_nrcvad_scores = Path('pseudo-gold/goldEN_with_NRC_VAD.vad')

1. Lets first evaluate against the Warriner scores

In [17]:
def compare_all(lexicon_name, best_models):
    results = []
    for m in best_models:
        lexicon_folder_level = []
        folder = Path(f'HistoricalVAD/{m[0]}_{m[1]}/')
        lexicon_folder_level.extend(sorted(list(folder.glob(f'{lexicon_name}[0-9]*'))))
        
        for lexicon_folder in lexicon_folder_level:
            results.append(compare_lexica(lexicon_folder, gold_standard_path_with_warriner_scores))
    return results

In [18]:
results_warriner = compare_all('Warriner', best_models_warriner)

index_names_warriner = [best_models_warriner[0][0]+best_models_warriner[0][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[1][0]+best_models_warriner[1][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[2][0]+best_models_warriner[2][1]+str(i*1000) for i in range(2,9)]

results_warriner_df = pd.DataFrame(results_warriner, columns=column_names, index=index_names_warriner) 
results_warriner_df

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextkNN2000,0.566752,0.0,0.475524,0.000016,0.438042,0.000190,0.493440
FastTextkNN3000,0.560662,0.0,0.498970,0.000002,0.459670,0.000022,0.506436
FastTextkNN4000,0.588638,0.0,0.515606,0.000000,0.477572,0.000020,0.527272
FastTextkNN5000,0.616510,0.0,0.542520,0.000000,0.490870,0.000000,0.549976
FastTextkNN6000,0.630496,0.0,0.564806,0.000000,0.514594,0.000000,0.569962
FastTextkNN7000,0.641740,0.0,0.578412,0.000000,0.521196,0.000000,0.580452
FastTextkNN8000,0.653184,0.0,0.577026,0.000000,0.523568,0.000000,0.584594
FastTextLinearRegression2000,0.627722,0.0,0.447456,0.000248,0.509912,0.000000,0.528364
FastTextLinearRegression3000,0.647904,0.0,0.483224,0.000000,0.530550,0.000000,0.553898
FastTextLinearRegression4000,0.664184,0.0,0.497600,0.000000,0.546548,0.000000,0.569452


In [ ]:
results_warriner_df.iloc[:7].to_csv(Path('pseudo-gold/Correlations_Warriner_pseudo-gold/Summary_FastText_kNN_Warriner.csv'))
results_warriner_df.iloc[7:14].to_csv(Path('pseudo-gold/Correlations_Warriner_pseudo-gold/Summary_FastText_LinReg_Warriner.csv')) 
results_warriner_df.iloc[14:].to_csv(Path('pseudo-gold/Correlations_Warriner_pseudo-gold/Summary_SGNS_LinReg_Warriner.csv')) 

In [11]:
results_nrc_vad = compare_all('NRC_VAD', best_models_nrc_vad)

index_names_nrc_vad_0 = [best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'10000']+[best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'0'+str(i*1000) for i in range(2,10)]
index_names_nrc_vad_1 = [best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'10000']+[best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'0'+str(i*1000) for i in range(2,10)] 
index_names_nrc_vad_2 = [best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'10000']+[best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'0'+str(i*1000) for i in range(2,10)] 

index_names_nrc_vad = index_names_nrc_vad_0 + index_names_nrc_vad_1 + index_names_nrc_vad_2

results_nrc_vad_df = pd.DataFrame(results_nrc_vad, columns=column_names, index=index_names_nrc_vad) 
results_nrc_vad_df 

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextLinearRegression10000,0.670900,0.0,0.517376,0.000000,0.365438,0.000240,0.517904
FastTextLinearRegression02000,0.620278,0.0,0.455172,0.000040,0.335026,0.001920,0.470154
FastTextLinearRegression03000,0.641174,0.0,0.481922,0.000000,0.347678,0.000900,0.490262
FastTextLinearRegression04000,0.653222,0.0,0.499982,0.000000,0.355182,0.000494,0.502804
FastTextLinearRegression05000,0.658826,0.0,0.502614,0.000000,0.360424,0.000428,0.507280
FastTextLinearRegression06000,0.665364,0.0,0.509424,0.000000,0.360088,0.000342,0.511624
FastTextLinearRegression07000,0.665096,0.0,0.512938,0.000000,0.360226,0.000332,0.512758
FastTextLinearRegression08000,0.666218,0.0,0.512916,0.000000,0.363414,0.000276,0.514178
FastTextLinearRegression09000,0.669046,0.0,0.516874,0.000000,0.365592,0.000250,0.517170
SGNSLinearRegression10000,0.683276,0.0,0.521192,0.000000,0.343568,0.000576,0.516022


In [ ]:
results_nrc_vad_df.iloc[:9].to_csv(Path('pseudo-gold/Correlations_Warriner_pseudo-gold/Summary_FastText_LinReg_NRC_VAD.csv'))
results_nrc_vad_df.iloc[9:18].to_csv(Path('pseudo-gold/Correlations_Warriner_pseudo-gold/Summary_SGNS_LinReg_NRC_VAD.csv')) 
results_nrc_vad_df.iloc[18:].to_csv(Path('pseudo-gold/Correlations_Warriner_pseudo-gold/Summary_SGNS_PaRaSim_NRC_VAD.csv')) 

2. Lets now evaluate against the NRC-VAD scores

In [12]:
def compare_all(lexicon_name, best_models):
    results = []
    for m in best_models:
        lexicon_folder_level = []
        folder = Path(f'HistoricalVAD/{m[0]}_{m[1]}/')
        lexicon_folder_level.extend(sorted(list(folder.glob(f'{lexicon_name}[0-9]*'))))
        
        for lexicon_folder in lexicon_folder_level:
            results.append(compare_lexica(lexicon_folder, gold_standard_path_with_nrcvad_scores))
    return results

In [13]:
results_warriner = compare_all('Warriner', best_models_warriner)

index_names_warriner = [best_models_warriner[0][0]+best_models_warriner[0][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[1][0]+best_models_warriner[1][1]+str(i*1000) for i in range(2,9)] + [best_models_warriner[2][0]+best_models_warriner[2][1]+str(i*1000) for i in range(2,9)]

results_warriner_df = pd.DataFrame(results_warriner, columns=column_names, index=index_names_warriner) 
results_warriner_df

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextkNN2000,0.582392,0.000000,0.404808,0.001164,0.187932,0.146024,0.391708
FastTextkNN3000,0.582266,0.000004,0.423744,0.000476,0.202210,0.098078,0.402742
FastTextkNN4000,0.608654,0.000000,0.421000,0.000120,0.204224,0.083906,0.411292
FastTextkNN5000,0.629548,0.000000,0.450838,0.000016,0.219186,0.058228,0.433190
FastTextkNN6000,0.640430,0.000000,0.480544,0.000002,0.208716,0.063904,0.443236
FastTextkNN7000,0.643904,0.000000,0.486248,0.000000,0.181548,0.088306,0.437236
FastTextkNN8000,0.653636,0.000000,0.478252,0.000000,0.177834,0.090140,0.436572
FastTextLinearRegression2000,0.662512,0.000000,0.498180,0.000006,0.288940,0.011052,0.483216
FastTextLinearRegression3000,0.687106,0.000000,0.538710,0.000000,0.303964,0.004384,0.509926
FastTextLinearRegression4000,0.703238,0.000000,0.554916,0.000000,0.309096,0.003238,0.522424


In [ ]:
results_warriner_df.iloc[:7].to_csv(Path('pseudo-gold/Correlations_NRCVAD_pseudo-gold/Summary_FastText_kNN_Warriner.csv'))
results_warriner_df.iloc[7:14].to_csv(Path('pseudo-gold/Correlations_NRCVAD_pseudo-gold/Summary_FastText_LinReg_Warriner.csv')) 
results_warriner_df.iloc[14:].to_csv(Path('pseudo-gold/Correlations_NRCVAD_pseudo-gold/Summary_SGNS_LinReg_Warriner.csv')) 

In [14]:
results_nrc_vad = compare_all('NRC_VAD', best_models_nrc_vad)

index_names_nrc_vad_0 = [best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'10000']+[best_models_nrc_vad[0][0]+best_models_nrc_vad[0][1]+'0'+str(i*1000) for i in range(2,10)]
index_names_nrc_vad_1 = [best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'10000']+[best_models_nrc_vad[1][0]+best_models_nrc_vad[1][1]+'0'+str(i*1000) for i in range(2,10)] 
index_names_nrc_vad_2 = [best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'10000']+[best_models_nrc_vad[2][0]+best_models_nrc_vad[2][1]+'0'+str(i*1000) for i in range(2,10)] 

index_names_nrc_vad = index_names_nrc_vad_0 + index_names_nrc_vad_1 + index_names_nrc_vad_2

results_nrc_vad_df = pd.DataFrame(results_nrc_vad, columns=column_names, index=index_names_nrc_vad) 
results_nrc_vad_df 

,r_V,p_value_V,r_A,p_value_A,r_D,p_value_D,r_mean
FastTextLinearRegression10000,0.761622,0.0,0.595024,0.0,0.676982,0.000000,0.677876
FastTextLinearRegression02000,0.706466,0.0,0.532840,0.0,0.631412,0.000000,0.623566
FastTextLinearRegression03000,0.728604,0.0,0.558706,0.0,0.647622,0.000000,0.644982
FastTextLinearRegression04000,0.742682,0.0,0.570870,0.0,0.659702,0.000000,0.657754
FastTextLinearRegression05000,0.752526,0.0,0.578462,0.0,0.667668,0.000000,0.666220
FastTextLinearRegression06000,0.755484,0.0,0.590120,0.0,0.670682,0.000000,0.672096
FastTextLinearRegression07000,0.756060,0.0,0.589716,0.0,0.672104,0.000000,0.672624
FastTextLinearRegression08000,0.757850,0.0,0.590028,0.0,0.674732,0.000000,0.674214
FastTextLinearRegression09000,0.759218,0.0,0.593528,0.0,0.675162,0.000000,0.675968
SGNSLinearRegression10000,0.755498,0.0,0.669228,0.0,0.706032,0.000000,0.710248


In [ ]:
results_nrc_vad_df.iloc[:9].to_csv(Path('pseudo-gold/Correlations_NRCVAD_pseudo-gold/Summary_FastText_LinReg_NRC_VAD.csv'))
results_nrc_vad_df.iloc[9:18].to_csv(Path('pseudo-gold/Correlations_NRCVAD_pseudo-gold/Summary_SGNS_LinReg_NRC_VAD.csv')) 
results_nrc_vad_df.iloc[18:].to_csv(Path('pseudo-gold/Correlations_NRCVAD_pseudo-gold/Summary_SGNS_PaRaSim_NRC_VAD.csv')) 